# Structural analysis of epistatic effects (Figure 5 and Supplemental Figure 9) in Tharp et al, 2026

**Analyses:**
- Alpha-carbon distance vs absolute sum of epistatic coefficients that include each mutation pair (Figure 5a for BA.1, expression, polyspecificity; Supplemental Figure 9a for Wuhan and BA.4).
- Buried surface area (ChimeraX on PDB 7ZFE) vs absolute sum of epistatic coefficients per Omi32 mutation (Figure 5b / Supplemental Figure 9b).
- Only coefficients that had significant bonferroni-corrected p-values were used (where p < 0.05/number of coefficients inferred for the model)

Figure 5c–d and Supplemental Figure 9c–e analyses were performed in ChimeraX (see Methods in Tharp et al, 2026) and Adobe Illustrator was used for final figure generation.

In [1]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path
import matplotlib.pyplot as plt
import copy
import scipy
from collections import OrderedDict
import seaborn as sns
from matplotlib.ticker import FixedLocator, FixedFormatter
from scipy.special import comb
from matplotlib.colors import LogNorm
from matplotlib.patches import Patch
import matplotlib as mpl
import matplotlib.lines as lines

plt.rcParams.update({'font.size': 8})
plt.rcParams.update({'font.family': 'sans-serif'})
plt.rcParams['font.sans-serif'] = "Arial"
plt.rcParams.update({'xtick.labelsize': 7})
plt.rcParams.update({'ytick.labelsize': 7})
plt.rcParams['scatter.edgecolors'] = 'black'
plt.rcParams['axes.linewidth'] = 0.5

# Run this notebook with working directory = Figures/Figure_5-S_Figure_9
REPO_ROOT = Path('.').resolve().parent.parent
LM = REPO_ROOT / 'epistasis_inference' / 'linear_interaction_models'

# Reference-based model coefficient files
PHENOTYPES = [
    {'path': LM / 'wuhan' / 'reference-based' / 'model_coeffs' / 'wuhan_raw_2order_full_biochem.txt', 'file_slug': 'wuhan'},
    {'path': LM / 'ba1' / 'reference-based' / 'model_coeffs' / 'ba1_raw_3order_full_biochem.txt', 'file_slug': 'ba1'},
    {'path': LM / 'ba4' / 'reference-based' / 'model_coeffs' / 'ba4_raw_3order_full_biochem.txt', 'file_slug': 'ba4'},
    {'path': LM / 'expression' / 'reference-based' / 'model_coeffs' / 'expression_raw_2order_full_biochem.txt', 'file_slug': 'expression'},
    {'path': LM / 'psr' / 'reference-based' / 'model_coeffs' / 'psr_raw_2order_full_biochem.txt', 'file_slug': 'polyspecificity'},
]


def load_coefficients_table(filepath: Path) -> pd.DataFrame:
    """Load biochem model_coeffs TSV; Bonferroni significance (same as Figure_3b / S_Figure_6-7)."""
    if not filepath.exists():
        raise FileNotFoundError(filepath)
    rows = []
    with open(filepath, 'r') as f:
        reader = csv.reader(f, delimiter='\t')
        _ = int(next(reader)[1])
        _ = float(next(reader)[1])
        _ = float(next(reader)[1])
        alpha_bonf = float(next(reader)[1])
        next(reader)
        for row in reader:
            if len(row) < 6:
                continue
            term = row[0]
            coef, se, pval = float(row[1]), float(row[2]), float(row[3])
            ci_l, ci_u = float(row[4]), float(row[5])
            is_sig = (term != 'Intercept' and pval < alpha_bonf)
            rows.append({
                'Term': term, 'Coefficient': coef, 'StdErr': se, 'p_value': pval,
                'CI_lower': ci_l, 'CI_upper': ci_u, 'is_significant': is_sig,
            })
    return pd.DataFrame(rows)


def figure_image_stem(file_slug: str) -> str:
    """Base name fragment for PNGs: {slug}_Figure_5 or {slug}_S_Figure_9."""
    if file_slug in ('ba1', 'expression', 'polyspecificity'):
        return f'{file_slug}_Figure_5'
    return f'{file_slug}_S_Figure_9'

In [2]:
# Pairwise epistasis vs alpha-carbon distance

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations
from collections import defaultdict
from scipy.stats import pearsonr
from pathlib import Path

try:
    from pymol import cmd
except ImportError as e:
    raise ImportError(
        "PyMOL is required for this cell. Install the conda environment in this folder and activate it with `conda activate structural-analysis`"
    ) from e

OUT_CSV_DIR = Path("abs_sum_distance/output")
OUT_IMG_DIR = Path("abs_sum_distance/images")
OUT_TXT_DIR = Path("abs_sum_distance/top_pair_terms")

OUT_CSV_DIR.mkdir(parents=True, exist_ok=True)
OUT_IMG_DIR.mkdir(parents=True, exist_ok=True)
OUT_TXT_DIR.mkdir(parents=True, exist_ok=True)

# load structure
cmd.reinitialize()
cmd.delete("all")
cmd.load("pdbs/7ZFE.pdb", "omi32")
cmd.hide("everything")

# mutation definitions
# PyMOL id -> (chain, residue number)
mutations = {
    '1':  ("M", 1),
    '2':  ("M", 31),
    '3':  ("M", 51),
    '4':  ("M", 56),
    '5':  ("M", 59),
    '6':  ("M", 88),
    '7':  ("N", 1),
    '8':  ("N", 3),
    '9':  ("N", 4),
    '10': ("N", 29),
    '11': ("N", 33),
    '12': ("N", 94),
    '13': ("N", 107),
}

def site_label(site_id):
    chain, resi = mutations[site_id]
    return f"{chain}{resi}"

all_distances = []

for cfg in PHENOTYPES:
    epistasis_file = cfg['path']
    stem = epistasis_file.stem
    epi = load_coefficients_table(epistasis_file)

    pair_epistasis = defaultdict(float)

    for _, row in epi.iterrows():
        if not row["is_significant"]:
            continue

        coef = float(row["Coefficient"])
        term = str(row["Term"])

        sites = [s.strip() for s in term.split(',')]
        sites = [s for s in sites if s in mutations]

        for i, j in combinations(sorted(sites), 2):
            pair_epistasis[(i, j)] += abs(coef)

    if not pair_epistasis:
        continue

    for (i, j), epi_val in pair_epistasis.items():
        sel_i = f"chain {mutations[i][0]} and resi {mutations[i][1]}"
        sel_j = f"chain {mutations[j][0]} and resi {mutations[j][1]}"

        d = cmd.distance(
            None,
            f"omi32 and ({sel_i}) and name CA",
            f"omi32 and ({sel_j}) and name CA"
        )
        all_distances.append(d)

global_x_max = max(all_distances)
global_xlim = (5, global_x_max + 0.12 * global_x_max)

print(f"Global x-axis range: [{global_xlim[0]:.2f}, {global_xlim[1]:.2f}]")

for cfg in PHENOTYPES:
    epistasis_file = cfg['path']
    file_slug = cfg['file_slug']
    stem = epistasis_file.stem
    print(f"Processing {stem}")

    epi = load_coefficients_table(epistasis_file)

    pair_epistasis = defaultdict(float)
    pair_terms = defaultdict(list)

    for _, row in epi.iterrows():
        if not row["is_significant"]:
            continue

        coef = float(row["Coefficient"])
        term = str(row["Term"])

        sites = [s.strip() for s in term.split(',')]
        sites = [s for s in sites if s in mutations]

        for i, j in combinations(sorted(sites), 2):
            pair_epistasis[(i, j)] += abs(coef)
            pair_terms[(i, j)].append((term, coef))

    if not pair_epistasis:
        print("  No significant pairwise signal found, skipping.")
        continue

    # compute alpha-carbon distances
    rows = []

    for (i, j), epi_val in pair_epistasis.items():
        sel_i = f"chain {mutations[i][0]} and resi {mutations[i][1]}"
        sel_j = f"chain {mutations[j][0]} and resi {mutations[j][1]}"

        d = cmd.distance(
            None,
            f"omi32 and ({sel_i}) and name CA",
            f"omi32 and ({sel_j}) and name CA"
        )

        rows.append({
            "pair": f"{site_label(i)},{site_label(j)}",
            "distance": d,
            "epistasis": epi_val,
            "i": i,
            "j": j
        })

    df = pd.DataFrame(rows).dropna()

    df.sort_values(by="epistasis", ascending=False).to_csv(
        OUT_CSV_DIR / f"{stem}_omi32_pair_epistasis_all_orders.csv",
        index=False
    )

    # report top pair by absolute sum of coefficients with that pair
    top_row = df.sort_values("epistasis", ascending=False).iloc[0]
    top_pair = (top_row.i, top_row.j)

    txt_out = OUT_TXT_DIR / f"{stem}_omi32_top_pair_{site_label(top_pair[0])}_{site_label(top_pair[1])}.txt"

    with open(txt_out, "w") as f:
        f.write(f"Dataset: {stem}\n")
        f.write(f"Top pair: {site_label(top_pair[0])}, {site_label(top_pair[1])}\n")
        f.write(f"Distance (Å): {top_row.distance:.2f}\n")
        f.write(f"Σ |coef|: {top_row.epistasis:.4g}\n\n")
        f.write("Contributing significant terms:\n")
        f.write("-" * 50 + "\n")

        for term, coef in sorted(
            pair_terms[top_pair],
            key=lambda x: abs(x[1]),
            reverse=True
        ):
            f.write(f"{term:30s}  coef = {coef:+.4g}\n")

    df_s = df.sort_values("epistasis", ascending=False).reset_index(drop=True)
    lim = 7

    print(f"\nTop 4 points for {stem}:")
    for i in range(min(4, len(df_s))):
        print(f"  {i+1}. {df_s.pair[i]:10s}  x={df_s.distance[i]:.2f}  y={df_s.epistasis[i]:.4f}")
    print()

    phenotype_labels = {
        "ba1": "BA.1",
        "ba4": "BA.4",
        "wuhan": "Wuhan",
        "polyspecificity": "polyspecificity",
        "psr": "polyspecificity",
        "expression": "expression",
    }
    
    phenotype_colors = {
        "polyspecificity": "#8B008B",
        "psr": "#8B008B",
        "wuhan": "#DC143C",
        "ba1": "#228B22",
        "ba4": "#0000CD",
        "expression": "#000000",
    }
    
    phenotype = None
    point_color = "black"  
    is_polyspecificity = False
    for key in phenotype_labels:
        if key in stem.lower():
            phenotype = phenotype_labels[key]
            point_color = phenotype_colors[key]
            if key in ["polyspecificity", "psr"]:
                is_polyspecificity = True
            break
    
    if phenotype is None:
        phenotype = stem 

    plt.rcParams.update({
        "font.family": "Arial",
        "axes.linewidth": 0.25,
        "axes.labelsize": 8,
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
        "xtick.major.pad": 1,
        "ytick.major.pad": 1,
        "axes.labelpad": 2,
    })

    # Create plot twice - once with labels, once without
    for include_labels in [False, True]:
        fig, ax = plt.subplots(figsize=(1.25, 0.9))
        fig.patch.set_alpha(0.0)  
        ax.patch.set_alpha(0.0)  

        sns.scatterplot(
            x=df_s.distance,
            y=df_s.epistasis,
            s=20,
            alpha=0.55,
            linewidth=0.4,
            edgecolor="black",
            legend=False,
            color=point_color,
            ax=ax
        )

        if len(df_s) >= 2:
            sns.regplot(
                x=df_s.distance,
                y=df_s.epistasis,
                scatter=False,
                ci=None,
                line_kws=dict(color="gray", lw=0.8),
                ax=ax
            )
            r, p = pearsonr(df_s.distance, df_s.epistasis)
        else:
            r, p = float("nan"), float("nan")

        if include_labels:
            x_range = df_s.distance.max() - df_s.distance.min()
            y_range = df_s.epistasis.max() - df_s.epistasis.min()
            
            offset_patterns = [
                (0.06, 0.10),   
                (0.06, -0.10),  
                (0.12, 0.05),   
                (0.12, -0.05),  
                (0.18, 0.12),   
                (0.18, -0.12),  
                (0.24, 0.00),   
            ]
            
            for i in range(min(lim, len(df_s))):
                x_offset, y_offset = offset_patterns[i]
                
                ax.text(
                    df_s.distance[i] + x_offset * x_range,
                    df_s.epistasis[i] + y_offset * y_range,
                    df_s.pair[i],
                    fontsize=6,
                    ha="left",
                    va="center",
                    fontfamily="Arial"
                )

      
        ax.text(
            0.95, 0.95,
            f"$r$ = {r:.2f}",
            transform=ax.transAxes,
            ha="right",
            va="top",
            fontsize=7,
            fontfamily="Arial"
        )

        ax.set_xlim(global_xlim)
        
        y = df_s.epistasis.values
        ax.set_ylim(min(y.min() - 0.10*(y.max()-y.min()), 0), y.max() + 0.15*(y.max()-y.min()))

        from matplotlib.ticker import FormatStrFormatter
        ax.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))

        ax.set_ylabel(f"{phenotype}\nΣ |coef| for pair")

        if is_polyspecificity:
            ax.set_xlabel(r"C$\mathrm{_\alpha}$–C$\mathrm{_\alpha}$ distance (Å)")
        else:
            ax.set_xlabel("")
            ax.set_xticklabels([])

        img_stem = figure_image_stem(file_slug)
        if include_labels:
            png_name = f"{img_stem}a_labeled.png"
        else:
            png_name = f"{img_stem}a.png"
        plt.savefig(
            OUT_IMG_DIR / png_name,
            dpi=2000,
            bbox_inches="tight",
            transparent=True
        )
        plt.close(fig)

Global x-axis range: [5.00, 33.20]
Processing wuhan_raw_2order_full_biochem

Top 4 points for wuhan_raw_2order_full_biochem:
  1. M56,M59     x=9.72  y=0.4139
  2. N33,M56     x=22.93  y=0.3491
  3. M51,M56     x=6.36  y=0.3290
  4. M56,N1      x=24.59  y=0.3080

Processing ba1_raw_3order_full_biochem

Top 4 points for ba1_raw_3order_full_biochem:
  1. M51,M56     x=6.36  y=2.2737
  2. M56,M59     x=9.72  y=1.6072
  3. N94,M51     x=15.96  y=1.5787
  4. N94,M56     x=18.15  y=1.4935

Processing ba4_raw_3order_full_biochem

Top 4 points for ba4_raw_3order_full_biochem:
  1. N33,M59     x=19.16  y=2.5673
  2. N33,M56     x=22.93  y=2.2855
  3. M56,M59     x=9.72  y=1.9951
  4. N33,M31     x=21.39  y=1.1908

Processing expression_raw_2order_full_biochem

Top 4 points for expression_raw_2order_full_biochem:
  1. M51,M56     x=6.36  y=0.1332
  2. M31,M56     x=11.63  y=0.1210
  3. M56,M59     x=9.72  y=0.0985
  4. M31,M51     x=11.52  y=0.0927

Processing psr_raw_2order_full_biochem

Top 4 

# buried surface area vs sum abs coefficient

In [3]:
# buried surface area vs sum of absolute coefficients that include each individual mutation 
## buried surface area analysis performed with ChimeraX (see Methods in Tharp et al, 2026)

from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from matplotlib.colors import to_hex, LinearSegmentedColormap
from matplotlib.ticker import FormatStrFormatter

OUT_CSV_DIR = Path("buried_SA/output")
OUT_IMG_DIR = Path("buried_SA/images")
OUT_COLOR_DIR = Path("buried_SA/chimerax_colors")

OUT_CSV_DIR.mkdir(exist_ok=True)
OUT_IMG_DIR.mkdir(exist_ok=True)
OUT_COLOR_DIR.mkdir(exist_ok=True)

BURIEDAREA_TXT = Path("buried_SA/chimeraX_buriedarea_7ZFE.txt")

# Chain letters in the 7ZFE structure file
HEAVY_CHAIN = "M"
LIGHT_CHAIN = "N"

# site_id -> (H/L, residue number)
mutations = {
    '1':  ("H", 1),
    '2':  ("H", 31),
    '3':  ("H", 51),
    '4':  ("H", 56),
    '5':  ("H", 59),
    '6':  ("H", 88),
    '7':  ("L", 1),
    '8':  ("L", 3),
    '9':  ("L", 4),
    '10': ("L", 29),
    '11': ("L", 33),
    '12': ("L", 94),
    '13': ("L", 107),
}

def parse_chimerax_buriedarea(txt_path: Path) -> dict:
    """Parse 7ZFE buried area file -> {(chain, resi): buried_area}"""
    pat = re.compile(
        r"Buried area between\s+/[A-Za-z0-9]+\s+and\s+/([A-Za-z0-9]):(\d+)\s*=\s*([0-9eE\.\+\-]+)"
    )
    return {
        (m.group(1), int(m.group(2))): float(m.group(3))
        for m in pat.finditer(txt_path.read_text())
    }

def build_site_bsa(lookup: dict) -> dict:
    """Map site_id -> buried_area using 7ZFE chain letters."""
    out = {}
    for site, (hl, resi) in mutations.items():
        chain = HEAVY_CHAIN if hl == "H" else LIGHT_CHAIN
        out[site] = lookup.get((chain, resi), np.nan)
    return out

def compute_site_effects(epistasis_path: Path):
    epi = load_coefficients_table(epistasis_path)
    total     = {s: 0.0 for s in mutations}
    total_abs = {s: 0.0 for s in mutations}

    for _, row in epi.iterrows():
        if not row['is_significant'] or pd.isna(row['Coefficient']):
            continue
        coef = float(row['Coefficient'])
        for s in str(row['Term']).split(','):
            s = s.strip()
            if s in mutations:
                total[s]     += coef
                total_abs[s] += abs(coef)

    return total, total_abs

# load buried surface area map (links each mutation to its buried surface area with SARS-CoV-2 BA.1 spike RBD from PDB 7ZFE)
bsa_lookup = parse_chimerax_buriedarea(BURIEDAREA_TXT)
site_bsa   = build_site_bsa(bsa_lookup)

all_bsa_values = [v for v in site_bsa.values() if not np.isnan(v)]
bsa_pad    = 0.05 * (max(all_bsa_values) - min(all_bsa_values))
global_xlim = (min(all_bsa_values) - bsa_pad, max(all_bsa_values) + bsa_pad)

plt.rcParams.update({
    "font.family":      "Arial",
    "axes.linewidth":   0.25,
    "axes.labelsize":   8,
    "xtick.labelsize":  7,
    "ytick.labelsize":  7,
    "xtick.major.pad":  1,
    "ytick.major.pad":  1,
    "axes.labelpad":    2,
})

phenotype_labels = {
    "ba1":            "BA.1",
    "ba4":            "BA.4",
    "wuhan":          "Wuhan",
    "polyspecificity": "polyspecificity",
    "psr":            "polyspecificity",
    "expression":     "expression",
}

phenotype_colors = {
    "polyspecificity": "#8B008B",
    "psr":             "#8B008B",
    "wuhan":           "#DC143C",
    "ba1":             "#228B22",
    "ba4":             "#0000CD",
    "expression":      "#000000",
}

for cfg in PHENOTYPES:
    epistasis_file = cfg['path']
    file_slug = cfg['file_slug']
    stem = epistasis_file.stem
    print(f"Processing {stem}")

    phenotype        = stem
    point_color      = "#000000"
    is_polyspecificity = False
    for key in phenotype_labels:
        if key in stem.lower():
            phenotype  = phenotype_labels[key]
            point_color = phenotype_colors[key]
            is_polyspecificity = key in ("polyspecificity", "psr")
            break

    total, total_abs = compute_site_effects(epistasis_file)

    df = pd.DataFrame({
        "site":         list(mutations),
        "chain":        [mutations[s][0] for s in mutations],
        "resi":         [mutations[s][1] for s in mutations],
        "buried_area":  [site_bsa[s]     for s in mutations],
        "total_effect": [total[s]        for s in mutations],
        "abs_effect":   [total_abs[s]    for s in mutations],
    }).dropna(subset=["buried_area"])

    df = df[df["abs_effect"] > 0]

    if len(df) < 2:
        print(f"  Skipping {stem}: fewer than 2 sites with significant effects.")
        continue

    cmap = LinearSegmentedColormap.from_list("custom", ["white", point_color], N=256)
    norm = plt.Normalize(vmin=df["abs_effect"].min(), vmax=df["abs_effect"].max())
    df["hex_color"] = df["abs_effect"].apply(lambda v: to_hex(cmap(norm(v))))

    df.to_csv(OUT_CSV_DIR / f"{stem}_bsa_vs_epistasis.csv", index=False)

    #generate ChimeraX color commands (for looking at mutations colored by absolute sum of coefficients in ChimeraX)
    with open(OUT_COLOR_DIR / f"{stem}_site_colors.txt", "w") as f:
        for r in df.itertuples():
            chain = HEAVY_CHAIN if r.chain == "H" else LIGHT_CHAIN
            f.write(f"color {r.hex_color} :{r.resi}.{chain}\n")

    for include_labels in [False, True]:
        fig, ax = plt.subplots(figsize=(1.25, 0.9))
        fig.patch.set_alpha(0.0)
        ax.patch.set_alpha(0.0)

        ax.scatter(
            df["buried_area"], df["abs_effect"],
            c=df["abs_effect"], cmap=cmap, norm=norm,
            s=20, linewidth=0.4, edgecolor="black", alpha=0.9, zorder=3
        )

        ax.set_xlim(global_xlim)

        if include_labels:
            x_range  = df["buried_area"].max() - df["buried_area"].min()
            y_range  = df["abs_effect"].max()  - df["abs_effect"].min()
            x_offset = 0.03 * x_range if x_range > 0 else 1.0
            y_jitter = 0.04 * y_range if y_range > 0 else 0.05
            rng = np.random.default_rng(seed=1)
            for r in df.itertuples():
                ax.text(
                    r.buried_area + x_offset,
                    r.abs_effect + rng.uniform(-y_jitter, y_jitter),
                    str(r.resi),
                    fontsize=5, ha="left", va="center",
                    color="black", zorder=10, fontfamily="Arial"
                )

        r_val, _ = pearsonr(df["buried_area"], df["abs_effect"])
        ax.text(0.95, 0.95, f"$r$ = {r_val:.2f}",
                transform=ax.transAxes, ha="right", va="top",
                fontsize=7, fontfamily="Arial")

        ax.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))
        ax.set_ylabel(f"{phenotype}\nΣ |coef| for mut")

        if is_polyspecificity:
            ax.set_xlabel("Buried area with antigen (Å²)")
        else:
            ax.set_xlabel("")
            ax.set_xticklabels([])

        img_stem = figure_image_stem(file_slug)
        if include_labels:
            png_name = f"{img_stem}b_labeled.png"
        else:
            png_name = f"{img_stem}b.png"
        plt.savefig(
            OUT_IMG_DIR / png_name,
            dpi=1000, bbox_inches="tight", transparent=True
        )
        plt.close(fig)

Processing wuhan_raw_2order_full_biochem
Processing ba1_raw_3order_full_biochem
Processing ba4_raw_3order_full_biochem
Processing expression_raw_2order_full_biochem
Processing psr_raw_2order_full_biochem
